# Gold EDA
Descriptive exploration of the Gold star schema (latest snapshot) needed for the business questions. Analysis only; not part of the weekly job.

## 1. Dimension exploration
Purpose: what values each dimension holds, the date ranges available and the age range of companies.
Tables: dim_company_classification, dim_status, dim_age_band, dim_company, fct_company_snapshot, fct_exposure.
Notes: ages are measured at the snapshot date (or at deregistration for deregistered companies).

In [0]:
SELECT * FROM workspace.sme_risk_gold.dim_company LIMIT 10

In [0]:
SELECT
    COUNT(*)                                   AS companies,
    COUNT_IF(registration_date IS NULL)        AS missing_registration_date,
    COUNT_IF(abn IS NULL)                      AS no_abn,
    COUNT_IF(previous_state IS NULL)           AS no_previous_state,
    COUNT_IF(state_registration_number IS NULL) AS no_state_reg_number,
    ROUND(AVG(CAST(has_abn AS INT)) * 100, 1)             AS pct_with_abn,
    ROUND(AVG(CAST(has_name_change AS INT)) * 100, 1)     AS pct_changed_name,
    ROUND(AVG(CAST(is_acn_named AS INT)) * 100, 2)        AS pct_acn_named,
    COUNT_IF(company_name_has_export_artifact)            AS names_with_export_artifact,
    MIN(as_of_snapshot_date)                   AS snapshot_date
FROM workspace.sme_risk_gold.dim_company

In [0]:
SELECT
    segment_sort_order,
    borrower_segment,
    COUNT(*) AS classification_combinations,
    concat_ws(', ', array_sort(collect_set(company_sub_class))) AS sub_classes
FROM workspace.sme_risk_gold.dim_company_classification
GROUP BY segment_sort_order, borrower_segment
ORDER BY segment_sort_order

In [0]:
SELECT status_sort_order, status_code, status_label, status_group, is_active, is_distressed, is_deregistered, in_population
FROM workspace.sme_risk_gold.dim_status
ORDER BY status_sort_order

In [0]:
SELECT acn, company_name, registration_date, former_name_count
FROM workspace.sme_risk_gold.dim_company
ORDER BY former_name_count DESC
LIMIT 10

In [0]:
SELECT age_band_key, age_band_label, min_age_years, max_age_years, age_group
FROM workspace.sme_risk_gold.dim_age_band
ORDER BY age_band_key

In [0]:
SELECT
    YEAR(current_name_start_date)                       AS name_change_year,
    COUNT(*)                                            AS companies_renamed
FROM workspace.sme_risk_gold.dim_company
WHERE current_name_start_date IS NOT NULL
GROUP BY 1
ORDER BY 1 DESC
LIMIT 15

In [0]:
SELECT
    (SELECT MIN(registration_date) FROM workspace.sme_risk_gold.dim_company)  AS earliest_registration,
    (SELECT MAX(registration_date) FROM workspace.sme_risk_gold.dim_company)  AS latest_registration,
    (SELECT MIN(to_date(CAST(window_start_date_key AS STRING), 'yyyyMMdd')) 
    FROM workspace.sme_risk_gold.fct_exposure) AS window_start,
    (SELECT MIN(to_date(CAST(snapshot_date_key AS STRING), 'yyyyMMdd'))     
    FROM workspace.sme_risk_gold.fct_exposure) AS window_end_snapshot,
    (SELECT DATEDIFF(MIN(to_date(CAST(snapshot_date_key AS STRING), 'yyyyMMdd')),
                     MIN(to_date(CAST(window_start_date_key AS STRING), 'yyyyMMdd')))
       FROM workspace.sme_risk_gold.fct_exposure) AS window_days

In [0]:
SELECT
    MIN(age_years) AS youngest_years,
    MAX(age_years)  AS oldest_years,
    PERCENTILE_DISC(0.5) WITHIN GROUP (ORDER BY age_years)     AS median_age_years,
    ROUND(AVG(age_years), 1)  AS average_age_years
FROM workspace.sme_risk_gold.fct_company_snapshot
WHERE is_active = 1 AND age_years IS NOT NULL

## 2. Measures exploration
Purpose: headline numbers for the latest snapshot, as a reference point for every later result.
Tables: fct_company_snapshot, fct_exposure, dim_company.
Notes: "deregistration rate" = deregistrations ÷ company-years at risk in the observation window (≈ last 12 months).

In [0]:
-- Find the total number of companies
SELECT COUNT(*) AS total_companies,
SUM(in_population) AS companies_in_population,
SUM(is_active) AS active_companies,
SUM(is_distressed) AS distressed_companies,
ROUND(100.0 * SUM(is_distressed) / SUM(is_active), 2) AS distress_share_pct,
SUM(is_deregistered) AS deregistered_last_12m
FROM workspace.sme_risk_gold.fct_company_snapshot;


In [0]:
-- Find the overall deregistration rate and the yearly chance of deregistering
SELECT
    ROUND(SUM(deregistrations) / SUM(exposure_years), 4)                 AS deregistrations_per_company_year,
    ROUND(100 * (1 - EXP(-SUM(deregistrations) / SUM(exposure_years))), 2) AS yearly_dereg_probability_pct
FROM workspace.sme_risk_gold.fct_exposure;

In [0]:
-- Find the share of companies with an ABN, ever renamed, and numbered (ACN-named)
SELECT
    ROUND(100.0 * SUM(CASE WHEN has_abn         THEN 1 ELSE 0 END) / COUNT(*), 1) AS pct_with_abn,
    ROUND(100.0 * SUM(CASE WHEN has_name_change THEN 1 ELSE 0 END) / COUNT(*), 1) AS pct_ever_renamed,
    ROUND(100.0 * SUM(CASE WHEN is_acn_named    THEN 1 ELSE 0 END) / COUNT(*), 2) AS pct_acn_named
FROM workspace.sme_risk_gold.dim_company;

In [0]:
-- Find the average and median age of active companies
SELECT
    ROUND(AVG(age_years), 1)                                AS avg_age_years,
    PERCENTILE_DISC(0.5) WITHIN GROUP (ORDER BY age_years)  AS median_age_years
FROM workspace.sme_risk_gold.fct_company_snapshot
WHERE is_active = 1;

In [0]:
-- Generate a report that shows all key metrics of the business
SELECT 'Total companies' AS measure_name, COUNT(*) AS measure_value
FROM workspace.sme_risk_gold.fct_company_snapshot
UNION ALL
SELECT 'Companies in population', SUM(in_population)
FROM workspace.sme_risk_gold.fct_company_snapshot
UNION ALL
SELECT 'Active companies', SUM(is_active)
FROM workspace.sme_risk_gold.fct_company_snapshot
UNION ALL
SELECT 'Distressed companies', SUM(is_distressed)
FROM workspace.sme_risk_gold.fct_company_snapshot
UNION ALL
SELECT 'Distress share of active (%)', ROUND(100.0 * SUM(is_distressed) / SUM(is_active), 2)
FROM workspace.sme_risk_gold.fct_company_snapshot
UNION ALL
SELECT 'Deregistered in last 12 months', SUM(is_deregistered)
FROM workspace.sme_risk_gold.fct_company_snapshot
UNION ALL
SELECT 'Company-years at risk', ROUND(SUM(exposure_years), 0)
FROM workspace.sme_risk_gold.fct_exposure
UNION ALL
SELECT 'Yearly deregistration probability (%)', ROUND(100 * (1 - EXP(-SUM(deregistrations) / SUM(exposure_years))), 2)
FROM workspace.sme_risk_gold.fct_exposure
UNION ALL
SELECT 'Companies with ABN (%)', ROUND(100.0 * SUM(CASE WHEN has_abn THEN 1 ELSE 0 END) / COUNT(*), 1)
FROM workspace.sme_risk_gold.dim_company
UNION ALL
SELECT 'Companies ever renamed (%)', ROUND(100.0 * SUM(CASE WHEN has_name_change THEN 1 ELSE 0 END) / COUNT(*), 1)
FROM workspace.sme_risk_gold.dim_company
UNION ALL
SELECT 'Median age of active companies (years)', PERCENTILE_DISC(0.5) WITHIN GROUP (ORDER BY age_years)
FROM workspace.sme_risk_gold.fct_company_snapshot
WHERE is_active = 1;

## 3. Magnitude analysis
Purpose: how many companies (and deregistrations) fall into each segment, status, age band and company attribute.
Tables: fct_company_snapshot, fct_exposure, dim_company_classification, dim_status, dim_age_band, dim_company.
Notes: counts only; risk rates come later in the business questions.

In [0]:
-- Find the total companies by borrower segment, split by status group
SELECT
    k.segment_sort_order,
    k.borrower_segment,
    COUNT(*)                AS total_companies,
    SUM(f.is_active)        AS active,
    SUM(f.is_distressed)    AS distressed,
    SUM(f.is_deregistered)  AS deregistered_last_12m
FROM workspace.sme_risk_gold.fct_company_snapshot f
LEFT JOIN workspace.sme_risk_gold.dim_company_classification k
    ON f.classification_key = k.classification_key
GROUP BY k.segment_sort_order, k.borrower_segment
ORDER BY k.segment_sort_order;

In [0]:
-- Find the total companies by status
SELECT
    s.status_sort_order,
    s.status_code,
    s.status_label,
    COUNT(*) AS total_companies
FROM workspace.sme_risk_gold.fct_company_snapshot f
LEFT JOIN workspace.sme_risk_gold.dim_status s
    ON f.status_key = s.status_key
GROUP BY s.status_sort_order, s.status_code, s.status_label
ORDER BY s.status_sort_order;

In [0]:
-- Find the total active companies by age band
SELECT
    a.age_band_key,
    a.age_band_label,
    COUNT(*) AS active_companies
FROM workspace.sme_risk_gold.fct_company_snapshot f
LEFT JOIN workspace.sme_risk_gold.dim_age_band a
    ON f.age_band_key = a.age_band_key
WHERE f.is_active = 1
GROUP BY a.age_band_key, a.age_band_label
ORDER BY a.age_band_key;

In [0]:
-- Find the deregistrations in the last 12 months by age band (age when they deregistered)
SELECT
    a.age_band_key,
    a.age_band_label,
    SUM(f.deregistrations) AS deregistrations
FROM workspace.sme_risk_gold.fct_exposure f
LEFT JOIN workspace.sme_risk_gold.dim_age_band a
    ON f.age_band_key = a.age_band_key
GROUP BY a.age_band_key, a.age_band_label
ORDER BY a.age_band_key;

In [0]:
-- Find the total companies with and without an ABN, by status group
SELECT
    CASE WHEN c.has_abn THEN 'Has ABN' ELSE 'No ABN' END  AS abn_held,
    COUNT(*)                AS total_companies,
    SUM(f.is_active)        AS active,
    SUM(f.is_distressed)    AS distressed,
    SUM(f.is_deregistered)  AS deregistered_last_12m
FROM workspace.sme_risk_gold.fct_company_snapshot f
LEFT JOIN workspace.sme_risk_gold.dim_company c
    ON f.company_key = c.company_key
GROUP BY 1
ORDER BY 1;

In [0]:
-- Find the total companies renamed vs never renamed, by status group
SELECT
    CASE WHEN c.has_name_change THEN 'Renamed' ELSE 'Never renamed' END  AS name_history,
    COUNT(*)                AS total_companies,
    SUM(f.is_active)        AS active,
    SUM(f.is_distressed)    AS distressed,
    SUM(f.is_deregistered)  AS deregistered_last_12m
FROM workspace.sme_risk_gold.fct_company_snapshot f
LEFT JOIN workspace.sme_risk_gold.dim_company c
    ON f.company_key = c.company_key
GROUP BY 1
ORDER BY 1;

In [0]:
-- Find the total numbered (ACN-named) vs named companies, by status group
SELECT
    CASE WHEN c.is_acn_named THEN 'Numbered (ACN in name)' ELSE 'Named' END  AS name_type,
    COUNT(*)                AS total_companies,
    SUM(f.is_active)        AS active,
    SUM(f.is_distressed)    AS distressed,
    SUM(f.is_deregistered)  AS deregistered_last_12m
FROM workspace.sme_risk_gold.fct_company_snapshot f
LEFT JOIN workspace.sme_risk_gold.dim_company c
    ON f.company_key = c.company_key
GROUP BY 1
ORDER BY 1;

In [0]:
-- Generate a report that shows company counts by every key dimension
WITH base AS (
    SELECT
        f.company_count,
        f.is_active,
        f.is_distressed,
        f.is_deregistered,
        k.borrower_segment,
        k.segment_sort_order,
        s.status_group,
        s.status_sort_order,
        a.age_band_label,
        a.age_band_key,
        CASE WHEN c.has_abn         THEN 'Has ABN'                ELSE 'No ABN'        END AS abn_held,
        CASE WHEN c.has_name_change THEN 'Renamed'                ELSE 'Never renamed' END AS name_history,
        CASE WHEN c.is_acn_named    THEN 'Numbered (ACN in name)' ELSE 'Named'         END AS name_type
    FROM workspace.sme_risk_gold.fct_company_snapshot f
    LEFT JOIN workspace.sme_risk_gold.dim_company_classification k ON f.classification_key = k.classification_key
    LEFT JOIN workspace.sme_risk_gold.dim_status                 s ON f.status_key         = s.status_key
    LEFT JOIN workspace.sme_risk_gold.dim_age_band               a ON f.age_band_key       = a.age_band_key
    LEFT JOIN workspace.sme_risk_gold.dim_company                c ON f.company_key        = c.company_key
),
report AS (
    SELECT 1 AS dimension_order, 'Borrower segment' AS dimension, borrower_segment AS category, segment_sort_order AS category_order,
           SUM(company_count) AS total_companies, SUM(is_active) AS active, SUM(is_distressed) AS distressed, SUM(is_deregistered) AS deregistered_last_12m
    FROM base GROUP BY borrower_segment, segment_sort_order
    UNION ALL
    SELECT 2, 'Status group', status_group, MIN(status_sort_order),
           SUM(company_count), SUM(is_active), SUM(is_distressed), SUM(is_deregistered)
    FROM base GROUP BY status_group
    UNION ALL
    SELECT 3, 'Age band', age_band_label, age_band_key,
           SUM(company_count), SUM(is_active), SUM(is_distressed), SUM(is_deregistered)
    FROM base GROUP BY age_band_label, age_band_key
    UNION ALL
    SELECT 4, 'ABN held', abn_held, CASE WHEN abn_held = 'Has ABN' THEN 1 ELSE 2 END,
           SUM(company_count), SUM(is_active), SUM(is_distressed), SUM(is_deregistered)
    FROM base GROUP BY abn_held
    UNION ALL
    SELECT 5, 'Name history', name_history, CASE WHEN name_history = 'Never renamed' THEN 1 ELSE 2 END,
           SUM(company_count), SUM(is_active), SUM(is_distressed), SUM(is_deregistered)
    FROM base GROUP BY name_history
    UNION ALL
    SELECT 6, 'Name type', name_type, CASE WHEN name_type = 'Named' THEN 1 ELSE 2 END,
           SUM(company_count), SUM(is_active), SUM(is_distressed), SUM(is_deregistered)
    FROM base GROUP BY name_type
)
SELECT
    dimension,
    category,
    total_companies,
    ROUND(100.0 * total_companies / SUM(total_companies) OVER (PARTITION BY dimension), 2) AS pct_of_companies,
    active,
    distressed,
    deregistered_last_12m
FROM report
ORDER BY dimension_order, category_order;

## 4. Change over time
Purpose: how deregistrations are spread across the observation window (monthly trend, spikes, batch days) and how today's companies are spread across registration years.
Tables: fct_company_snapshot, dim_date, dim_age_band.
Notes: the window runs 2025-10-20 → 2026-10-19, so October 2025 and October 2026 are partial months.
Registrations by year only count companies still in the file (survivorship), so they are not registration volumes.

In [0]:
-- Track deregistrations month by month across the observation window
SELECT
    year_month,
    deregistrations,
    deregistrations - LAG(deregistrations) OVER (ORDER BY year_month)   AS change_vs_previous_month,
    SUM(deregistrations) OVER (ORDER BY year_month)                     AS running_total
FROM (
    SELECT
        d.year_month,
        SUM(f.is_deregistered) AS deregistrations
    FROM workspace.sme_risk_gold.fct_company_snapshot f
    JOIN workspace.sme_risk_gold.dim_date d
        ON f.deregistration_date_key = d.date_key
    WHERE f.is_deregistered = 1
    GROUP BY d.year_month
) t
ORDER BY year_month;

In [0]:
-- Top 15 single dates by number of deregistrations
SELECT
    d.full_date,
    d.day_name,
    SUM(f.is_deregistered) AS deregistrations
FROM workspace.sme_risk_gold.fct_company_snapshot f
JOIN workspace.sme_risk_gold.dim_date d
    ON f.deregistration_date_key = d.date_key
WHERE f.is_deregistered = 1
GROUP BY d.full_date, d.day_name
ORDER BY deregistrations DESC
LIMIT 15;

In [0]:
-- Deregistrations by day of the week
SELECT
    d.day_name,
    SUM(f.is_deregistered)                                              AS deregistrations,
    ROUND(100.0 * SUM(f.is_deregistered) / SUM(SUM(f.is_deregistered)) OVER (), 1) AS pct
FROM workspace.sme_risk_gold.fct_company_snapshot f
JOIN workspace.sme_risk_gold.dim_date d
    ON f.deregistration_date_key = d.date_key
WHERE f.is_deregistered = 1
GROUP BY d.day_name
ORDER BY deregistrations DESC;

In [0]:
-- Monthly deregistrations split by age at deregistration
SELECT
    d.year_month,
    SUM(CASE WHEN a.age_band_key IN (1, 2) THEN 1 ELSE 0 END)  AS age_0_2,
    SUM(CASE WHEN a.age_band_key = 3       THEN 1 ELSE 0 END)  AS age_2_3,
    SUM(CASE WHEN a.age_band_key IN (4, 5) THEN 1 ELSE 0 END)  AS age_3_5,
    SUM(CASE WHEN a.age_band_key IN (6, 7) THEN 1 ELSE 0 END)  AS age_5_plus,
    COUNT(*)                                                   AS total
FROM workspace.sme_risk_gold.fct_company_snapshot f
JOIN workspace.sme_risk_gold.dim_date d     ON f.deregistration_date_key = d.date_key
JOIN workspace.sme_risk_gold.dim_age_band a ON f.age_band_key = a.age_band_key
WHERE f.is_deregistered = 1
GROUP BY d.year_month
ORDER BY d.year_month;

In [0]:
-- Survivorship: older years only include companies that still exist (plus last-year deregistrations)
SELECT
    d.financial_year_label,
    COUNT(*)                                                                AS companies,
    SUM(f.is_active)                                                        AS still_active,
    SUM(CASE WHEN d.month = 6 AND d.day_of_month >= 25 THEN 1 ELSE 0 END)   AS registered_25_30_june,
    ROUND(100.0 * SUM(CASE WHEN d.month = 6 AND d.day_of_month >= 25 THEN 1 ELSE 0 END) / COUNT(*), 1) AS pct_last_week_of_fy
FROM workspace.sme_risk_gold.fct_company_snapshot f
JOIN workspace.sme_risk_gold.dim_date d
    ON f.registration_date_key = d.date_key
WHERE d.financial_year >= 2007
GROUP BY d.financial_year_label
ORDER BY d.financial_year_label;

## 5. Part-to-whole analysis
Purpose: each group's share of the whole — companies vs deregistrations vs distress — to see which groups take more than their share.
Tables: fct_company_snapshot, fct_exposure, dim_company_classification, dim_age_band, dim_status.
Notes: over-representation = share of deregistrations ÷ share of exposure (company-years). Above 1 = deregisters more than its size suggests. This is a crude, not age-adjusted, comparison; the age-adjusted risk index comes in Q2.

In [0]:
WITH expo AS (
    SELECT k.borrower_segment, k.segment_sort_order,
           SUM(f.exposure_years)  AS exposure_years,
           SUM(f.deregistrations) AS deregistrations
    FROM workspace.sme_risk_gold.fct_exposure f
    JOIN workspace.sme_risk_gold.dim_company_classification k ON f.classification_key = k.classification_key
    GROUP BY k.borrower_segment, k.segment_sort_order
),
dist AS (
    SELECT k.borrower_segment,
           SUM(f.is_distressed) AS distressed
    FROM workspace.sme_risk_gold.fct_company_snapshot f
    JOIN workspace.sme_risk_gold.dim_company_classification k ON f.classification_key = k.classification_key
    GROUP BY k.borrower_segment
)
SELECT
    e.borrower_segment,
    ROUND(100.0 * e.exposure_years  / SUM(e.exposure_years)  OVER (), 2)  AS pct_of_exposure,
    ROUND(100.0 * e.deregistrations / SUM(e.deregistrations) OVER (), 2)  AS pct_of_deregistrations,
    ROUND(100.0 * d.distressed      / SUM(d.distressed)      OVER (), 2)  AS pct_of_distressed,
    ROUND((e.deregistrations / SUM(e.deregistrations) OVER ())
        / (e.exposure_years  / SUM(e.exposure_years)  OVER ()), 2)        AS over_representation
FROM expo e
JOIN dist d ON e.borrower_segment = d.borrower_segment
ORDER BY e.segment_sort_order;

In [0]:
SELECT
    a.age_band_key,
    a.age_band_label,
    ROUND(100.0 * SUM(f.exposure_years)  / SUM(SUM(f.exposure_years))  OVER (), 2) AS pct_of_exposure,
    ROUND(100.0 * SUM(f.deregistrations) / SUM(SUM(f.deregistrations)) OVER (), 2) AS pct_of_deregistrations,
    ROUND((SUM(f.deregistrations) / SUM(SUM(f.deregistrations)) OVER ())
        / (SUM(f.exposure_years)  / SUM(SUM(f.exposure_years))  OVER ()), 2)       AS over_representation
FROM workspace.sme_risk_gold.fct_exposure f
JOIN workspace.sme_risk_gold.dim_age_band a ON f.age_band_key = a.age_band_key
GROUP BY a.age_band_key, a.age_band_label
ORDER BY a.age_band_key;

In [0]:
SELECT
    k.borrower_segment,
    COUNT(*)                                                                         AS companies,
    ROUND(100.0 * SUM(CASE WHEN s.status_group = 'Registered'   THEN 1 ELSE 0 END) / COUNT(*), 2) AS pct_registered,
    ROUND(100.0 * SUM(CASE WHEN s.status_group = 'Distressed'   THEN 1 ELSE 0 END) / COUNT(*), 2) AS pct_distressed,
    ROUND(100.0 * SUM(CASE WHEN s.status_group = 'Not active'   THEN 1 ELSE 0 END) / COUNT(*), 2) AS pct_not_active,
    ROUND(100.0 * SUM(CASE WHEN s.status_group = 'Deregistered' THEN 1 ELSE 0 END) / COUNT(*), 2) AS pct_deregistered
FROM workspace.sme_risk_gold.fct_company_snapshot f
JOIN workspace.sme_risk_gold.dim_company_classification k ON f.classification_key = k.classification_key
JOIN workspace.sme_risk_gold.dim_status s                 ON f.status_key = s.status_key
GROUP BY k.borrower_segment, k.segment_sort_order
ORDER BY k.segment_sort_order;